# DLGenAI Project Milestone-4

Notebook adapted for the uploaded dataset (`train (1)(12).csv`).

This dataset contains the option columns **A, B, C, D, E** (instead of option_A...option_E).

## Install Required Libraries

In [1]:
!pip -q install transformers datasets peft accelerate

## Import Libraries

In [2]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, TaskType, get_peft_model
import torch.nn.functional as F

## Load Dataset

In [3]:
train_df = pd.read_csv('/content/train (1).csv')
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Q1. Label Encoding

In [4]:
label_map={"A":0,"B":1,"C":2,"D":3,"E":4}
train_df["label"]=train_df["answer"].map(label_map)
print("Q1 =", train_df.loc[150,"label"])

Q1 = 2


## Load Tokenizer

In [5]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Helper Function

In [6]:
option_cols=["A","B","C","D","E"]

def create_choices(row):
    return [str(row["prompt"])+" [SEP] "+str(row[col]) for col in option_cols]

## Q2. Prompt-Option Formatting

In [7]:
text=str(train_df.loc[0,"prompt"])+" [SEP] "+str(train_df.loc[0,"B"])
print("Q2 =", len(text))

Q2 = 407


## Q3. Single-Row MCQ Tokenization

In [8]:
choices=create_choices(train_df.loc[0])
enc=tokenizer(choices,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
input_ids=enc["input_ids"].unsqueeze(0)
print(input_ids.shape)
print("Q3 =", input_ids.shape[1])

torch.Size([1, 5, 128])
Q3 = 5


## Q4. Batch MCQ Tokenization

In [9]:
all_ids=[]
all_masks=[]

for i in range(16):
    enc=tokenizer(create_choices(train_df.loc[i]),padding="max_length",truncation=True,max_length=128,return_tensors="pt")
    all_ids.append(enc["input_ids"])
    all_masks.append(enc["attention_mask"])

input_ids=torch.stack(all_ids)
attention_mask=torch.stack(all_masks)

print(input_ids.shape)
print("Q4 =", input_ids.numel())

torch.Size([16, 5, 128])
Q4 = 10240


## Load Multiple-Choice Model

In [10]:
model=AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Q5. Multiple-Choice Logits

In [11]:
enc=tokenizer(create_choices(train_df.loc[0]),padding="max_length",truncation=True,max_length=128,return_tensors="pt")
outputs=model(input_ids=enc["input_ids"].unsqueeze(0),
              attention_mask=enc["attention_mask"].unsqueeze(0))
print(outputs.logits.shape)
print("Q5 =", outputs.logits.shape[1])

torch.Size([1, 5])
Q5 = 5


## Q6. Supervised Loss Tensor

In [12]:
label=torch.tensor([train_df.loc[0,"label"]])
outputs=model(input_ids=enc["input_ids"].unsqueeze(0),
              attention_mask=enc["attention_mask"].unsqueeze(0),
              labels=label)
print(outputs.loss)
print("Q6 =", outputs.loss.dim())

tensor(1.6009, grad_fn=<NllLossBackward0>)
Q6 = 0


## Q7. LoRA Trainable Parameters

In [15]:
!pip uninstall -y torchao
!pip install -q -U peft transformers accelerate datasets

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 6.4 MB/s eta 0:00:00


In [16]:
model=AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model=get_peft_model(model,config)

trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Q7 =", trainable)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q7 = 295681


## Q8. Hugging Face Dataset Preparation

In [17]:
small_df=train_df.iloc[:100].copy()
dataset=Dataset.from_pandas(small_df)

def preprocess(example):
    choices=[str(example["prompt"])+" [SEP] "+str(example[c]) for c in option_cols]
    enc=tokenizer(choices,padding="max_length",truncation=True,max_length=128)
    return {"input_ids":enc["input_ids"],
            "attention_mask":enc["attention_mask"],
            "labels":label_map[example["answer"]]}

dataset=dataset.map(preprocess)

print(torch.tensor(dataset[0]["input_ids"]).shape)
print("Q8 =", len(dataset[0]["input_ids"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

torch.Size([5, 128])
Q8 = 5


## Q9. Tiny LoRA Fine-Tuning

In [19]:
from datasets import Dataset
from transformers import (
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)
from peft import LoraConfig, TaskType, get_peft_model
import torch

# ----------------------------
# Create Dataset
# ----------------------------
tiny_df = train_df.iloc[:32].copy()
tiny_dataset = Dataset.from_pandas(tiny_df)

# ----------------------------
# Tokenization
# ----------------------------
def preprocess_train(example):
    choices = [
        str(example["prompt"]) + " [SEP] " + str(example[c])
        for c in option_cols
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": label_map[example["answer"]],
    }

tiny_dataset = tiny_dataset.map(preprocess_train)

# Remove unnecessary columns
cols_to_remove = [
    c for c in tiny_dataset.column_names
    if c not in ["input_ids", "attention_mask", "labels"]
]

tiny_dataset = tiny_dataset.remove_columns(cols_to_remove)

# ----------------------------
# Load Model
# ----------------------------
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

# ----------------------------
# Apply LoRA
# ----------------------------
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, config)

# Optional
model.print_trainable_parameters()

# ----------------------------
# Data Collator
# ----------------------------
data_collator = DefaultDataCollator(return_tensors="pt")

# ----------------------------
# Training Arguments
# ----------------------------
training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
    remove_unused_columns=False,
)

# ----------------------------
# Trainer
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tiny_dataset,
    data_collator=data_collator,
)

# ----------------------------
# Train
# ----------------------------
trainer.train()

print("Q9 =", trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.640072
2,1.601367
3,1.664974
4,1.592932


Q9 = 4


## Q10. Probability Assigned to Option E After Fine-Tuning

In [21]:
import torch
import torch.nn.functional as F

example = tiny_dataset[0]

# Convert lists to tensors
input_ids = torch.tensor(example["input_ids"], dtype=torch.long).unsqueeze(0)
attention_mask = torch.tensor(example["attention_mask"], dtype=torch.long).unsqueeze(0)

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probs = F.softmax(outputs.logits, dim=1)

print("Probabilities:", probs)
print("Q10 =", round(probs[0, 4].item(), 4))

Probabilities: tensor([[0.1999, 0.1882, 0.1978, 0.2056, 0.2085]])
Q10 = 0.2085
